In [ ]:
import json

# CẤU HÌNH
output_json_path = r'C:\FPTU\doangeo\data-synth\diagram_train_1k.json' # Tên file JSON mới
limit = 1000

# 1. Tái tạo lại danh sách đã sắp xếp (nếu bạn lỡ tắt biến sorted_metadata)
with open(r'C:\FPTU\doangeo\data-synth\diagram_train.json', 'r') as f:
    local_data = json.load(f)
    # Sort Y HỆT như lúc tải ảnh để đảm bảo khớp
    sorted_metadata = sorted(local_data, key=lambda x: x['image_id'])

# 2. Cắt lấy 1000 mẫu đầu tiên
subset_data = sorted_metadata[:limit]

# 3. Cập nhật đường dẫn ảnh (Optional)
print("Đang cập nhật đường dẫn ảnh trong JSON mới...")
for item in subset_data:
    # Gán đường dẫn mới chỉ là tên file
    item['image'] = f"{item['image_id']}.png"

# 4. Lưu file JSON mới
with open(output_json_path, 'w') as f:
    json.dump(subset_data, f, indent=4)

print(f"✅ Đã tạo file '{output_json_path}' với {len(subset_data)} mẫu.")

Đang cập nhật đường dẫn ảnh trong JSON mới...
✅ Đã tạo file 'C:\FPTU\doangeo\data-synth\diagram_train_1k.json' với 1000 mẫu.


In [5]:
import json
import re
from tqdm import tqdm

input_path = r'C:\FPTU\doangeo\data-synth\diagram_train_1k.json'
output_path = r'C:\FPTU\doangeo\data-synth\diagram_train_1k_vn.json'

# DANH SÁCH QUY TẮC DỊCH
translation_rules = [
    # 1. CỤM TỪ CỐ ĐỊNH & SỐ LƯỢNG
    ("two concentric circles", "hai đường tròn đồng tâm"),
    ("three concentric circles", "ba đường tròn đồng tâm"),
    ("concentric circles", "các đường tròn đồng tâm"),
    ("perpendicular bisector", "đường trung trực"),
    ("is perpendicular to", "vuông góc với"),
    ("is parallel to", "song song với"),
    ("is tangent to", "tiếp xúc với"),
    ("is similar to", "đồng dạng với"),
    ("is congruent to", "bằng"),
    ("is inside", "nằm trong"),
    ("is the", "là"),
    ("is on", "nằm trên"),

    # 2. CÁC THUẬT NGỮ DÀI
    ("equilateral triangle", "tam giác đều"),
    ("isosceles triangle", "tam giác cân"),
    ("right triangle", "tam giác vuông"),
    ("circumcircle", "đường tròn ngoại tiếp"),
    ("circumcenter", "tâm đường tròn ngoại tiếp"),
    ("incircle", "đường tròn nội tiếp"),
    ("incenter", "tâm đường tròn nội tiếp"),
    ("excircle", "đường tròn bàng tiếp"),
    ("excenter", "tâm đường tròn bàng tiếp"),
    ("parallelogram", "hình bình hành"),
    ("quadrilateral", "tứ giác"),
    ("rectangle", "hình chữ nhật"),
    ("trapezoid", "hình thang"),
    ("pentagon", "ngũ giác"),

    # 3. Extension, Centroid
    ("extension of", "phần kéo dài của"),
    ("extension", "phần kéo dài"),
    ("centroid of", "trọng tâm của"),
    ("centroid", "trọng tâm"),
    ("diagonal", "đường chéo"),
    ("orthocenter", "trực tâm"), 

    # 4. SỐ NHIỀU
    ("circles", "các đường tròn"),
    ("angles", "các góc"),
    ("segments", "các đoạn thẳng"),
    ("lines", "các đường thẳng"),
    ("rays", "các tia"),
    ("points", "các điểm"),

    # 5. SỐ ÍT
    ("circle", "đường tròn"),
    ("angle", "góc"),
    ("segment", "đoạn thẳng"),
    ("line", "đường thẳng"),
    ("ray", "tia"),
    ("point", "điểm"),
    ("diameter", "đường kính"),
    ("radius", "bán kính"),
    ("triangle", "tam giác"),
    ("square", "hình vuông"),

    # 6. TỪ NỐI & GIỚI TỪ
    ("intersects", "cắt"),
    ("intersect", "cắt"),
    ("through", "đi qua"),
    ("midpoint of", "trung điểm của"),
    ("with", "với"),
    ("and", "và"),
    ("of", "của"),
    ("at", "tại"),

    # BỔ SUNG QUAN TRỌNG
    ("is", "là"),  
    ("are", "là"),  

    # 7. SỐ ĐẾM
    ("two", "hai"),
    ("three", "ba"),
    ("one", "một")
]

# Ký tự đặc biệt
symbol_map = {
    "⊥": "vuông góc với",
    "//": "song song với"
}

def translate_final(text):
    # BƯỚC 1: XỬ LÝ MẠO TỪ "A"
    text = re.sub(r'\ba\s+(?![A-Z])', '', text)
    text = re.sub(r'^A\s+(?=[a-z])', '', text)

    # BƯỚC 2: THAY THẾ THEO DANH SÁCH
    for en_term, vn_term in translation_rules:
        pattern = re.compile(r'\b' + re.escape(en_term) + r'\b', re.IGNORECASE)
        text = pattern.sub(vn_term, text)

    # BƯỚC 3: XỬ LÝ KÝ TỰ ĐẶC BIỆT
    for sym, val in symbol_map.items():
        text = text.replace(sym, val)

    # BƯỚC 4: HẬU KỲ NGỮ PHÁP
    text = re.sub(r'(hai|ba|bốn|năm)\s+các\s+', r'\1 ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(the|that)\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()

    if text:
        text = text[0].upper() + text[1:]

    return text

# THỰC THI
print(f"Đang đọc dữ liệu từ: {input_path}")
with open(input_path, 'r') as f:
    data = json.load(f)

print(f"Đang xử lý {len(data)} mẫu (Đã fix lỗi 'is')...")

for item in tqdm(data, desc="Translation Fix"):
    original = item['caption'][0]
    translated = translate_final(original)
    item['caption_vn'] = translated

# Lưu file
print(f"\nĐang lưu kết quả vào: {output_path}")
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)



Đang đọc dữ liệu từ: C:\FPTU\doangeo\data-synth\diagram_train_1k.json
Đang xử lý 1000 mẫu (Đã fix lỗi 'is')...


Translation Fix: 100%|██████████| 1000/1000 [00:00<00:00, 1007.26it/s]



Đang lưu kết quả vào: C:\FPTU\doangeo\data-synth\diagram_train_1k_vn.json


In [6]:
import json

input_path = "C:\FPTU\doangeo\data-synth\diagram_train_1k_vn.json"
output_path = "C:\FPTU\doangeo\data-synth\diagram_train_1k_vn_cleaned.json"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    item.pop("image_id", None)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Done!")


Done!


<>:3: SyntaxWarning: invalid escape sequence '\F'
<>:4: SyntaxWarning: invalid escape sequence '\F'
<>:3: SyntaxWarning: invalid escape sequence '\F'
<>:4: SyntaxWarning: invalid escape sequence '\F'
C:\Users\Quang\AppData\Local\Temp\ipykernel_1052\3353951814.py:3: SyntaxWarning: invalid escape sequence '\F'
  input_path = "C:\FPTU\doangeo\data-synth\diagram_train_1k_vn.json"
C:\Users\Quang\AppData\Local\Temp\ipykernel_1052\3353951814.py:4: SyntaxWarning: invalid escape sequence '\F'
  output_path = "C:\FPTU\doangeo\data-synth\diagram_train_1k_vn_cleaned.json"


In [ ]:
import json
import shutil
from pathlib import Path

#  PATHS 
json_path = r"C:\FPTU\doangeo\data-synth\diagram_train_1k_vn_cleaned.json"
image_root = Path(r"C:\FPTU\doangeo\data-synth\train")
output_dir = Path(r"C:\FPTU\doangeo\data-synth\train_1k")

output_dir.mkdir(parents=True, exist_ok=True)

#  LOAD JSON 
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

#  COLLECT IMAGE NAMES 
image_names = set(item["image"] for item in data if "image" in item)

#  COPY FILES 
missing = []
copied = 0

for img_name in image_names:
    src = image_root / img_name
    dst = output_dir / img_name

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(img_name)

print(f"Copied: {copied} images")
print(f"Missing: {len(missing)} images")

if missing:
    print("Missing files:")
    for m in missing[:10]:
        print(" -", m)


Copied: 1000 images
Missing: 0 images
